## Notebook 11 - Player-level logistic regression: predicting match outcome
Núria Pascual Salas

**Content:** Multivariate logistic regression for the probability of winning a match,
based on player-level PageRank and controlling for position, minutes, and team style.
It tests whether PageRank centrality predicts winning and whether the effect differs
across positions. Standard errors are clustered by match-team, since players in the
same match share the same outcome and tactical context. PageRank is computed from the
match itself, so the model captures associations, not causal effects.

**Model:**
$$\mathrm{logit}\,P(\text{Win}) = \beta_0 + \beta_1 \mathrm{PR} + \beta_2 \mathrm{Pos}
+ \beta_3 (\mathrm{PR} \times \mathrm{Pos}) + \beta_4 \mathrm{Minutes}
+ \beta_5 \mathrm{TeamSF}$$

**Inputs:** player_match_matrix.csv (from notebook 06).

**Outputs:** outputs/csv/logistic_regression_results.csv

**Used in:** Chapter 5, Section 5.3 (player-level analysis).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import os

FIGURES_DIR = 'outputs/figures'
CSV_DIR     = 'outputs/csv'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

### 1. Load and prepare data

In [2]:
df = pd.read_csv(f'{CSV_DIR}/player_match_matrix.csv', encoding='utf-8-sig')

# Filter: remove UNK positions and players with very few minutes
df = df[df['position'] != 'UNK'].copy()
df = df[df['minutes_played'] >= 15].copy()
df = df.dropna(subset=['team_sf'])

print(f"Sample size after filtering: {len(df)}")
print(f"Outcome distribution:")
print(f"  Wins:   {df['won'].sum()} ({df['won'].mean()*100:.1f}%)")
print(f"  Not wins: {(1 - df['won']).sum()} ({(1 - df['won'].mean())*100:.1f}%)")

Sample size after filtering: 11128
Outcome distribution:
  Wins:   3952 (35.5%)
  Not wins: 7176 (64.5%)


### 2. Standardize continuous variables

In [3]:
for col in ['pagerank_pct', 'minutes_played', 'team_sf']:
    df[f'{col}_z'] = (df[col] - df[col].mean()) / df[col].std()

df['position'] = pd.Categorical(df['position'], categories=['GK', 'DEF', 'MID', 'FWD'])

print("Standardized variables (mean=0, std=1):")
print(df[['pagerank_pct_z', 'minutes_played_z', 'team_sf_z']].describe().round(3))

Standardized variables (mean=0, std=1):
       pagerank_pct_z  minutes_played_z  team_sf_z
count       11128.000         11128.000  11128.000
mean            0.000            -0.000      0.000
std             1.000             1.000      1.000
min            -1.832            -2.036     -2.129
25%            -0.831            -0.880     -0.742
50%            -0.075             0.348     -0.170
75%             0.719             0.854      0.707
max             3.689             1.251      3.207


### 3. Baseline model: PageRank only

In [4]:
# Cluster identifier: each (match_id, team_id) pair is a cluster
df['cluster_id'] = df['match_id'].astype(str) + '_' + df['team_id'].astype(str)

model_baseline = smf.logit('won ~ pagerank_pct_z', data=df).fit(
    disp=False,
    cov_type='cluster',
    cov_kwds={'groups': df['cluster_id']}
)
print(model_baseline.summary())

                           Logit Regression Results                           
Dep. Variable:                    won   No. Observations:                11128
Model:                          Logit   Df Residuals:                    11126
Method:                           MLE   Df Model:                            1
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:               0.0001412
Time:                        14:12:48   Log-Likelihood:                -7238.5
converged:                       True   LL-Null:                       -7239.6
Covariance Type:              cluster   LLR p-value:                    0.1527
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         -0.5966      0.076     -7.862      0.000      -0.745      -0.448
pagerank_pct_z     0.0283      0.010      2.762      0.006       0.008       0.048


### 4. Main effects model: PageRank + Position + Controls

In [5]:
model_main = smf.logit(
    'won ~ pagerank_pct_z + C(position) + minutes_played_z + team_sf_z',
    data=df
).fit(
    disp=False,
    cov_type='cluster',
    cov_kwds={'groups': df['cluster_id']}
)
print(model_main.summary())

                           Logit Regression Results                           
Dep. Variable:                    won   No. Observations:                11128
Model:                          Logit   Df Residuals:                    11121
Method:                           MLE   Df Model:                            6
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                 0.01701
Time:                        14:12:51   Log-Likelihood:                -7116.4
converged:                       True   LL-Null:                       -7239.6
Covariance Type:              cluster   LLR p-value:                 2.581e-50
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -0.6039      0.082     -7.400      0.000      -0.764      -0.444
C(position)[T.DEF]     0.0272      0.026      1.048      0.295      -0.024       0.078
C(position)[T.MID]  

### 5. Interaction model: PageRank × Position

In [6]:
model_interaction = smf.logit(
    'won ~ pagerank_pct_z * C(position) + minutes_played_z + team_sf_z',
    data=df
).fit(
    disp=False,
    cov_type='cluster',
    cov_kwds={'groups': df['cluster_id']}
)
print(model_interaction.summary())

                           Logit Regression Results                           
Dep. Variable:                    won   No. Observations:                11128
Model:                          Logit   Df Residuals:                    11118
Method:                           MLE   Df Model:                            9
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                 0.01905
Time:                        14:12:53   Log-Likelihood:                -7101.6
converged:                       True   LL-Null:                       -7239.6
Covariance Type:              cluster   LLR p-value:                 3.386e-54
                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -0.5775      0.157     -3.683      0.000      -0.885      -0.270
C(position)[T.DEF]                    0.0487      0.138      0

### 6. Compare the three models

In [7]:
df_compare = pd.DataFrame({
    'Model':           ['Baseline (PR only)', 'Main effects', 'Interaction'],
    'N parameters':    [len(model_baseline.params), len(model_main.params), len(model_interaction.params)],
    'Log-likelihood':  [model_baseline.llf,         model_main.llf,         model_interaction.llf],
    'AIC':             [model_baseline.aic,         model_main.aic,         model_interaction.aic],
    'Pseudo R²':       [model_baseline.prsquared,   model_main.prsquared,   model_interaction.prsquared],
}).round(3)
print(df_compare.to_string(index=False))
print()
print("Lower AIC = better balance of fit and parsimony.")
print("Pseudo R² values >0.20 are considered very good for logistic regression.")

             Model  N parameters  Log-likelihood       AIC  Pseudo R²
Baseline (PR only)             2       -7238.528 14481.056      0.000
      Main effects             7       -7116.420 14246.840      0.017
       Interaction            10       -7101.615 14223.230      0.019

Lower AIC = better balance of fit and parsimony.
Pseudo R² values >0.20 are considered very good for logistic regression.


### 7. Predictive performance: AUC and accuracy

In [8]:
# In-sample predictions for the interaction model
y_true     = df['won'].values
y_pred_prob = model_interaction.predict(df)
y_pred_bin  = (y_pred_prob >= 0.5).astype(int)

auc      = roc_auc_score(y_true, y_pred_prob)
accuracy = accuracy_score(y_true, y_pred_bin)
cm       = confusion_matrix(y_true, y_pred_bin)

print(f"Predictive performance (Interaction model, in-sample):")
print(f"  AUC:      {auc:.3f}")
print(f"  Accuracy: {accuracy:.3f}")
print()
print("Confusion matrix:")
print(f"                 Predicted 0   Predicted 1")
print(f"  Actual 0:      {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"  Actual 1:      {cm[1,0]:5d}        {cm[1,1]:5d}")
print()

Predictive performance (Interaction model, in-sample):
  AUC:      0.596
  Accuracy: 0.652

Confusion matrix:
                 Predicted 0   Predicted 1
  Actual 0:       7106           70
  Actual 1:       3802          150



### 8. Odds ratios with 95% confidence intervals

In [9]:
# Odds ratios = exp(coefficients) for easier interpretation
params = model_interaction.params
conf   = model_interaction.conf_int()
conf['OR']      = np.exp(params)
conf['OR_low']  = np.exp(conf[0])
conf['OR_high'] = np.exp(conf[1])
conf['p_value'] = model_interaction.pvalues

df_or = conf[['OR', 'OR_low', 'OR_high', 'p_value']].round(4)
df_or.columns = ['Odds Ratio', 'OR 95% Low', 'OR 95% High', 'p-value']
print(df_or)

df_or.to_csv(f'{CSV_DIR}/logistic_regression_results.csv')

                                   Odds Ratio  OR 95% Low  OR 95% High  \
Intercept                              0.5613      0.4128       0.7633   
C(position)[T.DEF]                     1.0499      0.8012       1.3758   
C(position)[T.MID]                     0.9041      0.6953       1.1757   
C(position)[T.FWD]                     1.0537      0.7975       1.3923   
pagerank_pct_z                         1.0488      0.7553       1.4562   
pagerank_pct_z:C(position)[T.DEF]      0.8246      0.5868       1.1586   
pagerank_pct_z:C(position)[T.MID]      1.0541      0.7459       1.4896   
pagerank_pct_z:C(position)[T.FWD]      1.0768      0.7549       1.5357   
minutes_played_z                       1.0123      0.9724       1.0539   
team_sf_z                              0.7214      0.6129       0.8490   

                                   p-value  
Intercept                           0.0002  
C(position)[T.DEF]                  0.7242  
C(position)[T.MID]                  0.4520  
C(pos

### Summary

In [10]:
print("=" * 70)
print("SUMMARY — Logistic regression for predicting Win")
print("=" * 70)
print()
print(f"Sample size: n = {len(df)}")
print(f"Outcome:     P(Win) = {df['won'].mean():.3f}")
print()
print("--- Model comparison (lower AIC = better) ---")
print(df_compare.to_string(index=False))
print()
print("--- Predictive performance (Interaction model) ---")
print(f"  AUC:      {auc:.3f}")
print(f"  Accuracy: {accuracy:.3f}")
print()
print("--- Significant predictors (p<0.05) ---")
sig_preds = df_or[df_or['p-value'] < 0.05]
if len(sig_preds) > 0:
    print(sig_preds.to_string())
else:
    print("  None at p<0.05 level")
print()

SUMMARY — Logistic regression for predicting Win

Sample size: n = 11128
Outcome:     P(Win) = 0.355

--- Model comparison (lower AIC = better) ---
             Model  N parameters  Log-likelihood       AIC  Pseudo R²
Baseline (PR only)             2       -7238.528 14481.056      0.000
      Main effects             7       -7116.420 14246.840      0.017
       Interaction            10       -7101.615 14223.230      0.019

--- Predictive performance (Interaction model) ---
  AUC:      0.596
  Accuracy: 0.652

--- Significant predictors (p<0.05) ---
           Odds Ratio  OR 95% Low  OR 95% High  p-value
Intercept      0.5613      0.4128       0.7633   0.0002
team_sf_z      0.7214      0.6129       0.8490   0.0001

